## Audio conversion (.mp3 to .wav)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install pydub

import os
from pydub import AudioSegment
from tqdm import tqdm

input_folder = "/content/drive/MyDrive/All_TTS_Data/Raw/Sanskrit_Datasets_24_02/vaksancayah_sanskrit_asr_corpus/Vāksañcayaḥ- Sanskrit_ASR_Corpus/sp020"
output_folder = "/content/drive/MyDrive/All_TTS_Data/Raw/Sanskrit_Datasets_24_02/Vanksancayah_sanskrit_asr_wav/sp020_wav"

os.makedirs(output_folder, exist_ok=True)

mp3_files = [f for f in os.listdir(input_folder) if f.endswith(".mp3")]

with tqdm(mp3_files, desc="Converting", unit="file") as pbar:
    for filename in pbar:
        pbar.set_postfix(file=filename[:30])  # Show current filename (truncated)

        mp3_path = os.path.join(input_folder, filename)
        wav_path = os.path.join(output_folder, filename.replace(".mp3", ".wav"))
        AudioSegment.from_mp3(mp3_path).export(wav_path, format="wav")


## Background Removing

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install audio-separator
!pip install onnxruntime

In [ ]:
import os
import sys
import shutil
import re
import json
import torch
from audio_separator.separator import Separator
from datetime import datetime
from tqdm import tqdm

CHECKPOINT_FILE = "/content/drive/MyDrive/All_TTS_Data/Raw/Conversational_checkpoints/Vanksancayah_sanskrit_asr/sp020.json"

def load_checkpoint():
    """Load checkpoint from JSON file"""
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, 'r') as f:
            return json.load(f)
    return {"processed_files": []}

def save_checkpoint(checkpoint):
    """Save checkpoint to JSON file"""
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump(checkpoint, f, indent=4)

def mark_as_processed(checkpoint, filepath):
    """Mark a file as processed in checkpoint"""
    if filepath not in checkpoint["processed_files"]:
        checkpoint["processed_files"].append(filepath)
        save_checkpoint(checkpoint)

def is_in_checkpoint(checkpoint, filepath):
    """Check if file is already in checkpoint"""
    return filepath in checkpoint["processed_files"]

def check_gpu():
    """Check and display GPU availability"""
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)} | CUDA: {torch.version.cuda} | PyTorch: {torch.__version__}\n")
        return True
    else:
        print("GPU not available, using CPU\n")
        return False

def get_user_input(prompt, is_path=False):
    """Get input from user with validation"""
    while True:
        user_input = input(prompt).strip()

        if not user_input:
            print("Input cannot be empty. Please try again.")
            continue

        if is_path and not os.path.exists(user_input):
            print(f"Path does not exist: {user_input}")
            continue

        return user_input

def get_optional_input(prompt, default_value):
    """Get optional input from user with default option"""
    while True:
        print(f"\n Default: {default_value}")
        choice = input(f"{prompt} (Press Enter to use default, or type 'custom' for custom input): ").strip().lower()

        if not choice or choice == '':
            return default_value
        elif choice == 'custom':
            custom_value = input("Enter custom value: ").strip()
            if custom_value:
                return custom_value
            else:
                print("Input cannot be empty. Please try again.")
                continue
        else:
            print("Invalid choice. Please press Enter for default or type 'custom'.")
            continue

def choose_processing_mode():
    """Let user choose between processing subdirectories or individual files"""
    while True:
        print("\nChoose processing mode:")
        print("   1. Process subdirectories (organized in folders)")
        print("   2. Process individual files (all files in one folder)")

        choice = input("\nEnter your choice (1 or 2): ").strip()

        if choice == '1':
            return 'subdirs'
        elif choice == '2':
            return 'individual'
        else:
            print("Invalid choice. Please enter 1 or 2.")

def extract_number(folder_name):
    """Extract number from folder name (e.g., 'mandal1' -> 1)"""
    match = re.search(r'\d+', folder_name)
    return int(match.group()) if match else float('inf')

def extract_sukta_number(filename):
    """Extract sukta number from filename"""
    match = re.search(r'sukta[_\s]*(\d+)', filename, re.IGNORECASE)
    return int(match.group(1)) if match else float('inf')

def is_already_processed(filename, output_folder):
    """Check if vocal file already exists in output folder"""
    base_name = os.path.splitext(filename)[0]
    expected_vocal_file = f"{base_name}_vocals.wav"
    output_path = os.path.join(output_folder, expected_vocal_file)
    return os.path.exists(output_path)

def setup_directories(input_folder, output_folder):
    """Verify input folder exists and create output folder"""
    if not os.path.exists(input_folder):
        print(f"Error: Input folder does not exist: {input_folder}")
        sys.exit(1)

    os.makedirs(output_folder, exist_ok=True)

def get_sorted_subdirs(input_folder):
    """Get and sort subdirectories numerically"""
    subdirs = [d for d in os.listdir(input_folder)
               if os.path.isdir(os.path.join(input_folder, d))]
    subdirs.sort(key=extract_number)
    return subdirs

def scan_processing_status(input_folder, output_folder, mode):
    """Compare input and output folders to show processing status"""
    if mode == 'subdirs':
        all_files = []
        subdirs = get_sorted_subdirs(input_folder)
        for subdir in subdirs:
            subdir_path = os.path.join(input_folder, subdir)
            wav_files = [f for f in os.listdir(subdir_path) if f.endswith('.wav')]
            all_files.extend(wav_files)
    else:
        all_files = [f for f in os.listdir(input_folder) if f.endswith('.wav')]

    already_processed = [f for f in all_files if is_already_processed(f, output_folder)]
    pending = [f for f in all_files if not is_already_processed(f, output_folder)]

    print("=" * 60)
    print("FOLDER COMPARISON REPORT")
    print(f"  Total input files    : {len(all_files)}")
    print(f"  Already processed    : {len(already_processed)}")
    print(f"  Pending processing   : {len(pending)}")
    print("=" * 60 + "\n")

def process_audio_file(separator, input_path, filename, output_folder, checkpoint, pbar=None):
    """Process a single audio file and extract vocals"""
    try:
        if is_already_processed(filename, output_folder):
            mark_as_processed(checkpoint, input_path)
            if pbar:
                pbar.set_postfix_str(f"Skipped: {filename}")
                pbar.update(1)
            return 'skipped'

        if is_in_checkpoint(checkpoint, input_path):
            if pbar:
                pbar.set_postfix_str(f"Skipped: {filename}")
                pbar.update(1)
            return 'skipped'

        base_name = os.path.splitext(filename)[0]
        output_name = f"{base_name}_vocals"

        output_names = {
            "Vocals": output_name
        }

        if pbar:
            pbar.set_postfix_str(f"Processing: {filename}")

        output_files = separator.separate(input_path, output_names)

        for file_path in output_files:
            if file_path and os.path.exists(file_path):
                if 'Vocals' in file_path or output_name in file_path:
                    vocal_filename = os.path.basename(file_path)
                    destination_path = os.path.join(output_folder, vocal_filename)
                    shutil.move(file_path, destination_path)
                else:
                    if os.path.exists(file_path):
                        os.remove(file_path)

        mark_as_processed(checkpoint, input_path)

        if pbar:
            pbar.set_postfix_str(f"Done: {filename}")
            pbar.update(1)

        return 'success'

    except Exception as e:
        if pbar:
            pbar.set_postfix_str(f"Failed: {filename}")
            pbar.update(1)
        tqdm.write(f"Error processing {filename}: {str(e)}")
        return 'failed'

def process_subdirectories_mode(input_folder, output_folder, separator, checkpoint):
    """Process audio files organized in subdirectories"""
    subdirs = get_sorted_subdirs(input_folder)

    if not subdirs:
        print("No subdirectories found in input folder")
        return

    # Collect all files across all subdirs
    all_wav_files = []
    for subdir in subdirs:
        subdir_path = os.path.join(input_folder, subdir)
        wav_files = [f for f in os.listdir(subdir_path) if f.endswith('.wav')]
        wav_files.sort(key=extract_sukta_number)
        for wav_file in wav_files:
            all_wav_files.append((subdir, wav_file))

    successful_files = 0
    failed_files = 0
    skipped_files = 0

    with tqdm(total=len(all_wav_files), desc="Processing audio", unit="file") as pbar:
        for subdir, wav_file in all_wav_files:
            subdir_path = os.path.join(input_folder, subdir)
            input_path = os.path.join(subdir_path, wav_file)
            pbar.set_description(f"Folder: {subdir}")
            result = process_audio_file(separator, input_path, wav_file, output_folder, checkpoint, pbar)

            if result == 'success':
                successful_files += 1
            elif result == 'skipped':
                skipped_files += 1
            else:
                failed_files += 1

    print_summary(len(subdirs), len(all_wav_files), successful_files, failed_files, skipped_files)

def process_individual_files_mode(input_folder, output_folder, separator, checkpoint):
    """Process individual audio files directly from input folder"""
    wav_files = [f for f in os.listdir(input_folder) if f.endswith('.wav')]
    wav_files.sort(key=extract_sukta_number)

    if not wav_files:
        print("No WAV files found in input folder")
        return

    successful_files = 0
    failed_files = 0
    skipped_files = 0

    with tqdm(total=len(wav_files), desc="Processing audio", unit="file") as pbar:
        for wav_file in wav_files:
            input_path = os.path.join(input_folder, wav_file)
            result = process_audio_file(separator, input_path, wav_file, output_folder, checkpoint, pbar)

            if result == 'success':
                successful_files += 1
            elif result == 'skipped':
                skipped_files += 1
            else:
                failed_files += 1

    print_summary(1, len(wav_files), successful_files, failed_files, skipped_files, is_individual=True)

def print_summary(total_folders, total_files, successful_files, failed_files, skipped_files, is_individual=False):
    """Print processing summary"""
    print("\n" + "=" * 60)
    print("PROCESSING COMPLETE")
    print("=" * 60)
    if not is_individual:
        print(f"Total folders processed  : {total_folders}")
    print(f"Total files found        : {total_files}")
    print(f"Skipped                  : {skipped_files}")
    print(f"Successfully processed   : {successful_files}")
    print(f"Failed                   : {failed_files}")
    print(f"Checkpoint saved         : {CHECKPOINT_FILE}")
    print(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 60)

def main():
    """Main execution function"""
    print("=" * 60)
    print("AUDIO VOCAL SEPARATOR (RESUME MODE)")
    print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 60 + "\n")

    check_gpu()

    checkpoint = load_checkpoint()
    already_done = len(checkpoint["processed_files"])
    if already_done > 0:
        print(f"Checkpoint loaded: {already_done} file(s) already processed\n")
    else:
        print("No checkpoint found, starting fresh\n")

    INPUT_FOLDER = get_user_input("Enter input folder path: ", is_path=True)
    OUTPUT_FOLDER = get_user_input("Enter output folder path: ")
    MODEL_FILE = "model_bs_roformer_ep_317_sdr_12.9755.ckpt"

    mode = choose_processing_mode()

    setup_directories(INPUT_FOLDER, OUTPUT_FOLDER)

    scan_processing_status(INPUT_FOLDER, OUTPUT_FOLDER, mode)

    print("Initializing separator...")
    try:
        separator = Separator()
        separator.load_model(model_filename=MODEL_FILE)
        print(f"Model loaded: {MODEL_FILE}\n")
    except Exception as e:
        print(f"Error loading model: {str(e)}")
        sys.exit(1)

    if mode == 'subdirs':
        process_subdirectories_mode(INPUT_FOLDER, OUTPUT_FOLDER, separator, checkpoint)
    else:
        process_individual_files_mode(INPUT_FOLDER, OUTPUT_FOLDER, separator, checkpoint)

if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        print(f"\nProcess interrupted. Progress saved in: {CHECKPOINT_FILE}")
        sys.exit(0)
    except Exception as e:
        print(f"\nFatal error: {str(e)}")
        sys.exit(1)

## Audio Duration Calculate

In [1]:
import os

FOLDERS = [
    r"\\IIPLSERVER\Data Engineering\TTS\Datasets\Malayalam\Male\Audio\indic_tts_ml_male",
    ]
INCLUDE_SUBFOLDERS = True
AUDIO_EXT = (".wav", ".mp3", ".flac")

for folder in FOLDERS:
    print(f"\n📁 {folder}")

    # 🔹 Collect files
    files = []
    walker = os.walk(folder) if INCLUDE_SUBFOLDERS else [(folder, [], os.listdir(folder))]
    for root, _, filenames in walker:
        files += [os.path.join(root, f) for f in filenames if f.lower().endswith(AUDIO_EXT)]

    print(f"   🎵 Files found: {len(files)}")


📁 \\IIPLSERVER\Data Engineering\TTS\Datasets\Malayalam\Male\Audio\indic_tts_ml_male
   🎵 Files found: 1907


In [2]:
import os
import soundfile as sf
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

FOLDERS = [
    r"/content/drive/MyDrive/All_TTS_Data/Audio_files_final/Conversational_Dataset/01_BG_Remove/English_Audio/Eng_Indian_male_01",
    ]

INCLUDE_SUBFOLDERS = True
AUDIO_EXT = (".wav", ".mp3", ".flac")

def format_duration(sec):
    return f"{int(sec//3600)} hr {int((sec%3600)//60)} min {int(sec%60)} sec"

def get_duration_fast(path):
    try:
        with sf.SoundFile(path) as f:
            return len(f) / f.samplerate
    except:
        return 0

grand_files, grand_duration = 0, 0

for folder in FOLDERS:
    print(f"\n📁 {folder}")

    # 🔹 Collect files
    files = []
    walker = os.walk(folder) if INCLUDE_SUBFOLDERS else [(folder, [], os.listdir(folder))]
    for root, _, filenames in walker:
        files += [os.path.join(root, f) for f in filenames if f.lower().endswith(AUDIO_EXT)]

    print(f"   🎵 Files found: {len(files)}")

    # 🔹 Parallel processing (FIXED INDENTATION)
    durations = []

    with ThreadPoolExecutor(max_workers=8) as executor:
        futures = {executor.submit(get_duration_fast, f): f for f in files}

        with tqdm(total=len(files), desc="⚡ Processing", unit="file", dynamic_ncols=True) as pbar:
            for future in as_completed(futures):
                file = futures[future]
                duration = future.result()

                durations.append(duration)

                # 🔥 Show current file (shortened for clean UI)
                pbar.set_postfix_str(os.path.basename(file)[:30])
                pbar.update(1)

    # 🔹 Folder summary
    folder_duration = sum(durations)
    print(f"   ✅ Duration: {format_duration(folder_duration)}")

    grand_files += len(files)
    grand_duration += folder_duration

# 🔹 Grand total
print(f"\n{'═'*45}")
print(f"  📊 TOTAL | Files: {grand_files} | {format_duration(grand_duration)}")
print(f"{'═'*45}")


📁 /content/drive/MyDrive/All_TTS_Data/Audio_files_final/Conversational_Dataset/01_BG_Remove/English_Audio/Eng_Indian_male_01
   🎵 Files found: 6661


⚡ Processing:   3%|▎         | 217/6661 [02:25<1:12:04,  1.49file/s, english_01_audio_005941.wav] 


KeyboardInterrupt: 

## Skip 2 & 2.5 Sec

In [ ]:
!df -h

Filesystem      Size  Used Avail Use% Mounted on
overlay         108G   30G   79G  28% /
tmpfs            64M     0   64M   0% /dev
shm             5.8G  4.0K  5.8G   1% /dev/shm
/dev/root       2.0G  1.2G  748M  63% /usr/sbin/docker-init
/dev/sda1       114G   31G   84G  27% /kaggle/input
tmpfs           6.4G  268K  6.4G   1% /var/colab
tmpfs           6.4G     0  6.4G   0% /proc/acpi
tmpfs           6.4G     0  6.4G   0% /proc/scsi
tmpfs           6.4G     0  6.4G   0% /sys/firmware
drive           108G   34G   75G  31% /content/drive


In [ ]:
import os
import soundfile as sf
from tqdm import tqdm

SUPPORTED_EXTENSIONS = {'.wav', '.flac', '.ogg', '.aiff', '.au', '.rf64'}
THRESHOLD = 2.0

folder = input("Enter folder path: ").strip()

# Collect all audio files first
all_files = []
for root, _, files in os.walk(folder):
    for f in files:
        if os.path.splitext(f)[1].lower() in SUPPORTED_EXTENSIONS:
            all_files.append(os.path.join(root, f))

# Scan with tqdm
short_files = []
with tqdm(total=len(all_files), desc="Scanning", unit="file", colour="cyan", dynamic_ncols=True) as pbar:
    for filepath in all_files:
        pbar.set_postfix(file=os.path.basename(filepath)[:40], refresh=False)
        try:
            duration = sf.info(filepath).duration
            if duration < THRESHOLD:
                short_files.append(filepath)
        except:
            pass
        pbar.update(1)

print(f"\nFound {len(short_files)} file(s) under {THRESHOLD}s")

if short_files:
    confirm = input(f"\nDelete all {len(short_files)} files? (yes/no): ").strip().lower()
    if confirm in ("yes", "y"):
        for f in short_files:
            os.remove(f)
        print(f"Deleted {len(short_files)} file(s).")
    else:
        print("Cancelled. No files deleted.")

Enter folder path: /content/drive/MyDrive/All_TTS_Data/Audio_files_final/Conversational_Dataset/01_BG_Remove/Hindi_Audio/abhirl_hindi_tts_dataset/abhirl_hindi_tts_dataset_ritu_f


Scanning: 100%|██████████| 456/456 [00:22<00:00, 20.01file/s, file=ritu_0912.wav] 


Found 0 file(s) under 2.0s


#File_Moving_Process

In [ ]:
import os, shutil
import tqdm as tqdm

src = r"/content/drive/MyDrive/All_TTS_Data/Audio_files_final/Conversational_Dataset/01_BG_Remove/Tamil_Audio/IndicTTS_Tamil/female/audio"
dst = r"/content/drive/MyDrive/All_TTS_Data/Audio_files_final/Conversational_Dataset/01_BG_Remove/Tamil_Audio/Indic_TTS_Tamil_female"

os.makedirs(dst, exist_ok=True)

os.makedirs(dst, exist_ok=True)

# 🔹 collect all files first
all_files = []
for root, _, files in os.walk(src):
    for f in files:
        all_files.append(os.path.join(root, f))

# 🔹 single progress bar
for path in tqdm(all_files, desc="⚡ Moving", unit="file", dynamic_ncols=True):
    dst_path = os.path.join(dst, os.path.basename(path))
    shutil.move(path, dst_path)

print("✅ Done")

TypeError: 'module' object is not callable. Did you mean: 'tqdm.tqdm(...)'?

#Create_Final_csv

# Comparing Audio with csv file

In [1]:
!pip install pandas

In [3]:
import os
import pandas as pd

# ==============================
# PATHS
# ==============================
audio_folder = r"\\IIPLSERVER\Data Engineering\TTS\Datasets\Marathi\Both\Audio\Marathi_20hr_data"
csv_path = r"\\IIPLSERVER\Data Engineering\TTS\Datasets\Marathi\Both\CSV\Marathi_20hr_data.csv"   # <-- change this

# ==============================
# LOAD DATA
# ==============================

# Get audio file names from folder
audio_files = set(os.listdir(audio_folder))

# Load CSV
df = pd.read_csv(csv_path)

# Extract only filename from full path in CSV
csv_files = set(df["Audio_path"].apply(lambda x: os.path.basename(x)))

# ==============================
# COMPARISON
# ==============================

# Files in folder but NOT in CSV
missing_in_csv = audio_files - csv_files

# Files in CSV but NOT in folder
missing_in_folder = csv_files - audio_files

# ==============================
# RESULTS
# ==============================

print("📁 Total audio files in folder:", len(audio_files))
print("📄 Total audio files in CSV:", len(csv_files))

print("\n❌ Missing in CSV:", len(missing_in_csv))
for file in sorted(missing_in_csv):
    print(file)

print("\n❌ Missing in Folder:", len(missing_in_folder))
for file in sorted(missing_in_folder):
    print(file)

📁 Total audio files in folder: 10869
📄 Total audio files in CSV: 9845

❌ Missing in CSV: 1024
MAR_M_ALEXA_00014.wav
MAR_M_ALEXA_00019.wav
MAR_M_ALEXA_00026.wav
MAR_M_ALEXA_00040.wav
MAR_M_ALEXA_00053.wav
MAR_M_ALEXA_00064.wav
MAR_M_ALEXA_00082.wav
MAR_M_ALEXA_00095.wav
MAR_M_ALEXA_00121.wav
MAR_M_ALEXA_00134.wav
MAR_M_ALEXA_00141.wav
MAR_M_ALEXA_00160.wav
MAR_M_ALEXA_00168.wav
MAR_M_ALEXA_00187.wav
MAR_M_ALEXA_00201.wav
MAR_M_ALEXA_00213.wav
MAR_M_ALEXA_00225.wav
MAR_M_ALEXA_00238.wav
MAR_M_ALEXA_00243.wav
MAR_M_ALEXA_00259.wav
MAR_M_ALEXA_00262.wav
MAR_M_ALEXA_00307.wav
MAR_M_ALEXA_00320.wav
MAR_M_ALEXA_00333.wav
MAR_M_ALEXA_00361.wav
MAR_M_ALEXA_00365.wav
MAR_M_ALEXA_00377.wav
MAR_M_ALEXA_00388.wav
MAR_M_ALEXA_00409.wav
MAR_M_ALEXA_00411.wav
MAR_M_ALEXA_00413.wav
MAR_M_ALEXA_00421.wav
MAR_M_ANGER_00007.wav
MAR_M_ANGER_00020.wav
MAR_M_ANGER_00047.wav
MAR_M_ANGER_00050.wav
MAR_M_ANGER_00056.wav
MAR_M_ANGER_00085.wav
MAR_M_ANGER_00093.wav
MAR_M_ANGER_00102.wav
MAR_M_ANGER_00125.wav
MAR_